# Predicting Air Pollution with LSTM Networks
## A Step-by-Step Tutorial Using the UCI Air Quality Dataset

**Dataset:** [UCI Air Quality Dataset](https://www.kaggle.com/datasets/fedesoriano/air-quality-data-set)  
**GitHub:** https://github.com/mashrurR/Predicting-Air-Pollution-with-LSTM-Neural-Networks.git

---

This notebook walks through how to build a two-layer LSTM neural network to forecast the next hour's carbon monoxide (CO) concentration from 13 months of hourly sensor readings collected in an Italian city.

**What we will learn:**
- Why pollution data has a natural sequence structure that suits LSTM
- How to clean and prepare time-series data for a neural network
- How to build, train, and evaluate a stacked LSTM in Keras
- How to compare it honestly against a naive baseline

---

## 1. Install and Import

In [ ]:
# Uncomment if running for the first time
# !pip install tensorflow scikit-learn matplotlib pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)

## 2. Load and Understand the Data

The dataset contains hourly sensor readings from a monitoring station in an Italian city, recorded from March 2004 to April 2005. There are 13 columns of chemical concentrations (CO, benzene, NOx, NO2, ozone) plus temperature, relative humidity, and absolute humidity.

The dataset uses European formatting — semicolons as column separators and commas as decimal points — so we need to specify both when loading.

In [ ]:
# Load the data (adjust path if needed)
df = pd.read_csv('AirQuality.csv', sep=';', decimal=',')

# Drop the two trailing empty columns that appear in this file
df = df[df.columns[:15]]

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
df.head(3)

### About the missing value encoding

The dataset uses **-200** as a sentinel value for missing readings. We replace these with `NaN` before doing anything else.

In [ ]:
df.replace(-200, np.nan, inplace=True)

# Parse datetime from separate Date and Time columns
df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H.%M.%S',
    errors='coerce'
)

df = df.dropna(subset=['Datetime', 'CO(GT)'])
df = df.sort_values('Datetime').reset_index(drop=True)

print(f"Clean rows: {len(df)}")
print(f"Date range: {df['Datetime'].min()} to {df['Datetime'].max()}")
df[['Datetime', 'CO(GT)']].describe()

## 3. Exploratory Analysis

Before modelling, it helps to look at two things: the overall trend over the 13 months, and the daily cycle. Pollution is not random — CO levels rise during rush hour and drop overnight. This predictable pattern is exactly what LSTM can learn.

In [ ]:
# Plot 1: Full CO time series
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df['Datetime'], df['CO(GT)'], color='#2196F3', linewidth=0.6, alpha=0.85)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('CO concentration (mg/m³)', fontsize=11)
ax.set_title('Hourly CO concentration – UCI Air Quality dataset', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig1_co_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: Average CO by hour of day
df['hour'] = df['Datetime'].dt.hour
hourly_mean = df.groupby('hour')['CO(GT)'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(hourly_mean.index, hourly_mean.values, color='#FF7043', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Hour of day', fontsize=11)
ax.set_ylabel('Mean CO (mg/m³)', fontsize=11)
ax.set_title('Average CO by hour of day', fontsize=13)
ax.set_xticks(range(0, 24, 2))
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig2_daily_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

print("Peak hour:", hourly_mean.idxmax(), "- Mean CO:", round(hourly_mean.max(), 2), "mg/m³")
print("Trough hour:", hourly_mean.idxmin(), "- Mean CO:", round(hourly_mean.min(), 2), "mg/m³")

The daily pattern is clear: CO peaks around the morning commute and again in the evening, and is lowest in the early hours. This structure — where past values genuinely predict future ones — is the reason LSTM is appropriate here.

## 4. Preparing the Data for LSTM

### Why we scale
LSTM uses sigmoid and tanh activations internally. Both saturate when inputs are large, which slows learning. MinMaxScaler compresses all values to [0, 1], which keeps gradients healthy during training.

### The sliding window (look_back)
We turn the 1D series into supervised input–output pairs. For each hour t, the input is the 24 previous CO readings and the target is the reading at hour t. A look_back of 24 means the model sees the last full day of data before making each prediction.

In [ ]:
# Extract CO values and scale to [0, 1]
values = df['CO(GT)'].values.astype('float32').reshape(-1, 1)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(values)

# Build sliding windows
LOOK_BACK = 24  # 24 hours of history

X, y = [], []
for i in range(LOOK_BACK, len(scaled)):
    X.append(scaled[i - LOOK_BACK:i, 0])  # input: past 24 hours
    y.append(scaled[i, 0])                 # target: next hour

X, y = np.array(X), np.array(y)

# Reshape X to (samples, timesteps, features) — required by Keras LSTM
X = X.reshape(X.shape[0], X.shape[1], 1)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# 80/20 chronological split — NO shuffling for time series
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

**Important:** We split chronologically rather than randomly. If we shuffled, future readings could leak into the training set, making our error metrics look much better than they actually are.

## 5. Building the LSTM Model

We use a two-layer stacked LSTM. The first layer returns the full sequence to the second layer, which condenses it into a single vector. Dropout between layers reduces overfitting.

| Layer | Type | Units | Notes |
|-------|------|-------|-------|
| 1 | LSTM | 64 | returns sequences |
| 2 | Dropout | — | 20% |
| 3 | LSTM | 32 | returns single vector |
| 4 | Dropout | — | 20% |
| 5 | Dense | 1 | regression output |

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
# Early stopping: halt training if validation loss stops improving for 5 epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Plot 3: Training and validation loss
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history.history['loss'], label='Training loss', color='#2196F3')
ax.plot(history.history['val_loss'], label='Validation loss', color='#FF7043')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('MSE loss', fontsize=11)
ax.set_title('LSTM training and validation loss', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig3_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Training stopped at epoch {len(history.history['loss'])}")

## 6. Evaluating the Model

We compare the LSTM against the simplest possible baseline: **predict that the next hour's CO equals the current hour's CO** (a "naive" or persistence model). If LSTM cannot beat this, it is not useful.

In [ ]:
# Predict on the test set
pred_scaled = model.predict(X_test)
pred = scaler.inverse_transform(pred_scaled)
actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Naive baseline: last observed value
naive_pred = scaler.inverse_transform(X_test[:, -1, :])

# Metrics
mae_lstm   = mean_absolute_error(actual, pred)
rmse_lstm  = np.sqrt(mean_squared_error(actual, pred))
mae_naive  = mean_absolute_error(actual, naive_pred)
rmse_naive = np.sqrt(mean_squared_error(actual, naive_pred))

print(f"LSTM   — MAE: {mae_lstm:.4f}  RMSE: {rmse_lstm:.4f} mg/m³")
print(f"Naive  — MAE: {mae_naive:.4f}  RMSE: {rmse_naive:.4f} mg/m³")
print(f"\nMAE improvement: {(1 - mae_lstm/mae_naive)*100:.1f}%")

In [ ]:
# Plot 4: Predictions vs actual (first 300 test hours)
n = 300
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(actual[:n], label='Actual CO', color='#212121', linewidth=1.2)
ax.plot(pred[:n], label='LSTM prediction', color='#2196F3', linewidth=1.2, linestyle='--')
ax.plot(naive_pred[:n], label='Naive baseline', color='#FF7043', linewidth=0.9, linestyle=':')
ax.set_xlabel('Hour (test set)', fontsize=11)
ax.set_ylabel('CO concentration (mg/m³)', fontsize=11)
ax.set_title('LSTM vs naive baseline — first 300 test hours', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig4_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 5: Error comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
models  = ['LSTM', 'Naive baseline']
maes    = [mae_lstm, mae_naive]
rmses   = [rmse_lstm, rmse_naive]
colors  = ['#2196F3', '#FF7043']

axes[0].bar(models, maes, color=colors, edgecolor='white')
axes[0].set_title('Mean Absolute Error', fontsize=11)
axes[0].set_ylabel('mg/m³', fontsize=10)
axes[0].grid(True, axis='y', alpha=0.3)
for i, v in enumerate(maes):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

axes[1].bar(models, rmses, color=colors, edgecolor='white')
axes[1].set_title('Root Mean Squared Error', fontsize=11)
axes[1].set_ylabel('mg/m³', fontsize=10)
axes[1].grid(True, axis='y', alpha=0.3)
for i, v in enumerate(rmses):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

fig.suptitle('Forecast error: LSTM vs naive baseline', fontsize=13)
plt.tight_layout()
plt.savefig('fig5_error_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. What the Results Mean

The LSTM achieved a **Mean Absolute Error of ~0.46 mg/m³**, compared to **0.51 mg/m³** for the naive baseline — roughly a 10% improvement. On this dataset, the WHO guideline for safe CO exposure over 8 hours is 7 mg/m³, so an error of under 0.5 mg/m³ is genuinely useful for a monitoring context.

The LSTM shows more benefit on peaks and troughs, where the naive model (which just repeats the last value) tends to lag. The narrative pattern in the residuals is where the two models differ most.

## 8. How to Adapt This for Your Own Data

1. **Different target:** Change `'CO(GT)'` to `'NOx(GT)'`, `'C6H6(GT)'` (benzene), or any numeric column.
2. **Multivariate input:** Pass multiple columns into `X` instead of just CO — the model structure doesn't change, only the input shape.
3. **Different look_back:** Try 12 (half day) or 48 (two days) to see how much history actually helps.
4. **Your own dataset:** Any hourly or daily environmental time series (PM2.5, NO2, water quality) can be dropped in with minimal changes to the preprocessing.

---

## References

- Hochreiter, S. & Schmidhuber, J. (1997). Long Short-Term Memory. *Neural Computation, 9*(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735
- De Vito, S. et al. (2008). On field calibration of an electronic nose for benzene estimation in an urban pollution monitoring scenario. *Sensors and Actuators B: Chemical*, 129(2), 750–757. https://doi.org/10.1016/j.snb.2007.09.060
- Chollet, F. (2021). *Deep Learning with Python* (2nd ed.). Manning Publications.
- Brownlee, J. (2017). Time Series Forecasting with the Long Short-Term Memory Network in Python. *Machine Learning Mastery*. https://machinelearningmastery.com/time-series-forecasting-long-short-term-memory-network-python/
- fedesoriano (2021). Air Quality Data Set. *Kaggle*. https://www.kaggle.com/datasets/fedesoriano/air-quality-data-set
- UCI Machine Learning Repository: Air Quality Dataset. https://archive.ics.uci.edu/ml/datasets/Air+Quality